In [9]:
import csv
import re

def parse_refer_format(file_path):
    """Parse refer format file and extract paper information."""
    papers = []
    current_paper = {}
    authors = []
    
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            
            # Empty line indicates end of a paper entry
            if not line:
                if current_paper:
                    # Join multiple authors
                    if authors:
                        current_paper['authors'] = "; ".join(authors)
                    papers.append(current_paper)
                    current_paper = {}
                    authors = []
                continue
            
            # Parse different fields
            if line.startswith('%T '):
                current_paper['title'] = line[3:].strip()
            elif line.startswith('%A '):
                authors.append(line[3:].strip())
            elif line.startswith('%B '):
                current_paper['venue'] = line[3:].strip()
            elif line.startswith('%D '):
                current_paper['year'] = line[3:].strip()
            elif line.startswith('%8 '):
                current_paper['publication_date'] = line[3:].strip()
            elif line.startswith('%U '):
                current_paper['url'] = line[3:].strip()
            elif line.startswith('%X '):
                # Abstract might span multiple lines
                abstract = line[3:].strip()
                current_paper['abstract'] = abstract
            elif not line.startswith('%') and 'abstract' in current_paper:
                # Continue previous abstract if line doesn't start with %
                current_paper['abstract'] += " " + line.strip()
        
        # Add last paper if exists
        if current_paper:
            if authors:
                current_paper['authors'] = "; ".join(authors)
            papers.append(current_paper)
    
    return papers

def save_to_csv(papers, output_path):
    """Save parsed papers to CSV file."""
    if not papers:
        print("No papers to save!")
        return
    
    # Define CSV columns
    fieldnames = ['title', 'authors', 'venue', 'year', 'publication_date', 'url', 'abstract']
    
    with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        
        for paper in papers:
            # Ensure all fields exist (fill with empty string if missing)
            row = {field: paper.get(field, '') for field in fieldnames}
            writer.writerow(row)
    
    print(f"✅ Saved {len(papers)} papers to {output_path}")

# Parse the file
input_file = "../data/search_results-refer.txt"
output_file = "../data/search_results.csv"

print(f"📖 Parsing {input_file}...")
papers = parse_refer_format(input_file)

print(f"📊 Found {len(papers)} papers")
print("\n🔍 Sample paper:")
if papers:
    sample = papers[0]
    for key, value in sample.items():
        print(f"  {key}: {value[:100] if isinstance(value, str) and len(value) > 100 else value}")

# Save to CSV
save_to_csv(papers, output_file)
print(f"\n✅ Done! CSV file saved to: {output_file}")

📖 Parsing ../data/search_results-refer.txt...
📊 Found 2998 papers

🔍 Sample paper:
  title: Foreword IRIS44: Living in a Digital World?
  venue: Selected Papers of the IRIS, Issue Nr 12 (2021)
  year: 2021
  publication_date: January  1, 2021
  url: https://aisel.aisnet.org/iris2021/1
  authors: Hochwarter, Stefan; Wik, Malin
✅ Saved 2998 papers to ../data/search_results.csv

✅ Done! CSV file saved to: ../data/search_results.csv


In [1]:
print("============")

In [1]:
import pandas as pd
from langchain_openai.embeddings import OpenAIEmbeddings
from neo4j import GraphDatabase
# from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv
import os
import time
load_dotenv()


True

In [2]:
# Initialize embedding model (DeepSeek via OpenAI-compatible API)
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    # openai_api_base="https://api.deepseek.com/v1",
    # api_key=os.getenv("DEEPSEEK_API_KEY")
)

In [3]:
# embedding_model = SentenceTransformer("Qwen/Qwen3-Embedding-8B")
print("=======================")
# Load the CSV file
df = pd.read_csv("../data/search_results.csv")
print(f"📊 Loaded {len(df)} papers from CSV")


📊 Loaded 2998 papers from CSV
📊 Loaded 2998 papers from CSV


In [4]:
# Prepare texts for embedding
print("\n� Preparing texts for embedding...")
texts_to_embed = []
for idx, row in df.iterrows():
    # Combine title and abstract for embedding
    text = f"Title: {row.get('title', '')}\n\nAbstract: {row.get('abstract', '')}"
    texts_to_embed.append(text)

print(f"Prepared {len(texts_to_embed)} texts")



� Preparing texts for embedding...
Prepared 2998 texts


In [5]:
# Create embeddings in batches using embed_documents
print("\n🔄 Generating embeddings in batches...")
BATCH_SIZE = 200  # Process 100 documents at a time (adjust based on your needs)
embeddings_list = []

try:
    for i in tqdm(range(0, len(texts_to_embed), BATCH_SIZE), desc="Embedding batches"):
        batch = texts_to_embed[i:i+BATCH_SIZE]
        
        # Use embed_documents for batch processing
        batch_embeddings = embedding_model.embed_documents(batch)
        embeddings_list.extend(batch_embeddings)
        
        # Optional: Add small delay between batches to avoid rate limits
        if i + BATCH_SIZE < len(texts_to_embed):
            time.sleep(0.5)  # 500ms delay between batches
        
except Exception as e:
    print(f"\n❌ Error during embedding: {e}")
    print(f"Successfully embedded {len(embeddings_list)} out of {len(texts_to_embed)} documents")
    
    # Fill remaining with zero vectors if error occurred
    if len(embeddings_list) < len(texts_to_embed):
        embedding_dim = len(embeddings_list[0]) if embeddings_list else 1536
        remaining = len(texts_to_embed) - len(embeddings_list)
        print(f"⚠️  Filling {remaining} remaining documents with zero vectors")
        for _ in range(remaining):
            embeddings_list.append([0.0] * embedding_dim)



🔄 Generating embeddings in batches...


Embedding batches: 100%|██████████| 15/15 [00:45<00:00,  3.04s/it]


In [7]:
# Add embeddings to dataframe
df['embedding'] = embeddings_list
print(f"✅ Generated {len(embeddings_list)} embeddings")
print(f"   Embedding dimension: {len(embeddings_list[0])}")

# Connect to Neo4j
print("\n🔗 Connecting to Neo4j...")
neo4j_uri = "bolt://localhost:7687"
neo4j_user = "neo4j"
neo4j_password = "123456789"

driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

def create_paper_node(tx, paper_data):
    """Create a Paper node in Neo4j with all properties."""
    query = """
    CREATE (p:Paper {
        title: $title,
        authors: $authors,
        venue: $venue,
        year: $year,
        publication_date: $publication_date,
        url: $url,
        abstract: $abstract,
        embedding: $embedding
    })
    RETURN p
    """
    tx.run(query, 
           title=paper_data.get('title', ''),
           authors=paper_data.get('authors', ''),
           venue=paper_data.get('venue', ''),
           year=paper_data.get('year', ''),
           publication_date=paper_data.get('publication_date', ''),
           url=paper_data.get('url', ''),
           abstract=paper_data.get('abstract', ''),
           embedding=paper_data['embedding'])

# Clear existing Paper nodes (optional)
print("🗑️  Clearing existing Paper nodes...")
with driver.session() as session:
    session.run("MATCH (p:Paper) DETACH DELETE p")

# Insert papers into Neo4j
print("\n📥 Loading papers into Neo4j...")
successful_inserts = 0

with driver.session() as session:
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Loading to Neo4j"):
        try:
            paper_data = {
                'title': str(row.get('title', '')),
                'authors': str(row.get('authors', '')),
                'venue': str(row.get('venue', '')),
                'year': str(row.get('year', '')),
                'publication_date': str(row.get('publication_date', '')),
                'url': str(row.get('url', '')),
                'abstract': str(row.get('abstract', '')),
                'embedding': row['embedding']
            }
            # Use write_transaction instead of execute_write for compatibility
            session.write_transaction(create_paper_node, paper_data)
            successful_inserts += 1
        except Exception as e:
            print(f"\n⚠️  Error inserting paper {idx}: {e}")

driver.close()
print(f"\n✅ Successfully loaded {successful_inserts}/{len(df)} papers into Neo4j")

# Verify the data
print("\n🔍 Verifying data in Neo4j...")
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
with driver.session() as session:
    result = session.run("MATCH (p:Paper) RETURN count(p) as count")
    count = result.single()['count']
    print(f"Total papers in Neo4j: {count}")
    
    # Show sample paper
    result = session.run("MATCH (p:Paper) RETURN p LIMIT 1")
    sample = result.single()
    if sample:
        print("\nSample paper:")
        paper = sample['p']
        print(f"  Title: {paper['title'][:100]}...")
        print(f"  Authors: {paper['authors'][:100]}...")
        print(f"  Year: {paper['year']}")
        print(f"  Embedding dimensions: {len(paper['embedding'])}")

driver.close()
print("\n🎉 Done!")

✅ Generated 2998 embeddings
   Embedding dimension: 1536

🔗 Connecting to Neo4j...
🗑️  Clearing existing Paper nodes...

📥 Loading papers into Neo4j...


Loading to Neo4j: 100%|██████████| 2998/2998 [00:24<00:00, 121.78it/s]




✅ Successfully loaded 2998/2998 papers into Neo4j

🔍 Verifying data in Neo4j...
Total papers in Neo4j: 2998

Sample paper:
  Title: Foreword IRIS44: Living in a Digital World?...
  Authors: Hochwarter, Stefan; Wik, Malin...
  Year: 2021
  Embedding dimensions: 1536

🎉 Done!


In [ ]:
query_text = """
Researcher A has expertise in time series, federated learning, machine learning. Here is his past research:
    - Privacy-preserving federated learning for residential short-term load forecasting
    - Secure federated learning for residential short term load forecasting
    - Bridging Smart Meter Gaps: A Benchmark of Statistical, Machine Learning and Time Series Foundation Models for Data Imputation
    - Towards a peer-to-peer residential short-term load forecasting with federated learning
    - Spatiotemporal Graph Neural Networks in short term load forecasting: Does adding Graph Structure in Consumption Data Improve Predictions?

    He wants to collaborate and do research on anomaly detection in mobile traffic 
    Please propose a research idea that suits his past research and match with the topic he want to contributes to. 
"""

query_embedding = embedding_model.embed_query(query_text)


In [9]:
user: str = "neo4j"
password: str = "123456789"
uri: str = "bolt://localhost:7687"
driver = GraphDatabase.driver(uri, auth=(user, password))    

In [10]:
cypher_query = """
                MATCH (p:Paper)
                WITH p, 
                     gds.similarity.cosine(p.embedding, $query_embedding) AS similarity
                WHERE similarity > 0.5
                RETURN p.title AS title,
                       p.authors AS authors,
                       p.abstract AS abstract,
                       p.year AS year,
                       p.venue AS venue,
                       p.url AS url,
                       similarity
                ORDER BY similarity DESC
                LIMIT $top_k
                """

result = session.run(
                    cypher_query,
                    query_embedding=query_embedding,
                    top_k=5
                )


In [11]:
papers = []
for record in result:
    papers.append({
        "title": record["title"],
        "authors": record["authors"],
        "abstract": record["abstract"],
        "year": record["year"],
        "venue": record["venue"],
        "url": record["url"],
        "similarity": round(record["similarity"], 3)
    })


In [14]:
papers

[{'title': 'A Framework for Model-Centric Cross-Silo Horizontal Federated Machine Learning',
  'authors': 'Mohamed, Haytham M.; El-Gayar, Omar',
  'abstract': 'Typically, in machine learning applications, the data is combined and centralized in one place alongside a model to train. This imposes the concern of exposing sensitive data and security risks (Mammen, 2021). Federated learning offers a better option to mitigate such challenges. In federated learning, data is distributed across different locations where a machine learning model is trained locally and only the results (not the data) are sent back to a centralized server to aggregate and enhance the trained model. The way in which data is split across the different locations matters in terms of how federated learning is implemented and the practical and technical challenges. The data sets in horizontal (or homogenous) federated learning preserve the same feature space but have different examples (Yang et al., 2019). Furthermore, 